### Installiamo `gurobipy`, documentazione @ https://www.gurobi.com/documentation/9.5/refman/py_python_api_details.html#sec:Python-details

In [10]:
!pip3 install gurobipy
!pip3 install scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 38.1 MB/s  0:00:00 eta 0:00:01


# Importiamo i pacchetti necessari

In [2]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np

# Volgiamo definire e risolvere il seguente problema

\begin{align}
  \text{maximize}\ &250x_1 + 230x_2 + 110x_3 + 350x_4 + 110x_5\\
                  &s.t.\\
                  &0\le\ x_i \lt \infty\\
                  &2x_1 + 1.5x_2 + 0.5x_3 + 2.5x_4 + 0.7x_5\leq 100.\\
                  &0.5x_1 + 0.25x_2 + 0.25x_3 + x_4 + 0.3x_5\leq 50\\
                  &x_1 + x_2 \geq 10\\
                  &x_3 = x_5\\
\end{align}

Creiamo il modello

In [2]:
model = gp.Model()

Restricted license - for non-production use only - expires 2027-11-29


Aggiungiamo le variabili con i rispettivi limiti

In [3]:
n_vars = 5
#con lb definisco il lower bound e con ub l'upper bound, con [0] * n_vars ottengo una lista di zeri lunga n_vars 
x = model.addMVar(n_vars, lb=[0] * n_vars, ub=GRB.INFINITY)
model.update()

In [4]:
print(f"Model vars:{model.getVars()}\nNum vars: {len(model.getVars())}\nVar C1: {model.getVarByName('C0')}")

Model vars:[<gurobi.Var C0>, <gurobi.Var C1>, <gurobi.Var C2>, <gurobi.Var C3>, <gurobi.Var C4>]
Num vars: 5
Var C1: <gurobi.Var C0>


Aggiungiamo i vincoli del problema

In [5]:
A =  np.array([[2., 1.5, 0.5, 2.5, 0.7],

               [0.5, 0.25, 0.25, 1, 0.3],
               [1, 1, 0, 0, 0],
               [0, 0, 1, 0, -1]])

b = np.array([100, 50, 10, 0])
ct = model.addConstr(A[:2]@x <= b[:2])
ct2 = model.addConstr(A[-2]@x >= b[-2])
ct3 = model.addConstr(A[-1]@x == b[-1])

model.update()

Definiamo la funzione obiettivo

In [6]:
obj_coefs = np.array([250, 230, 110, 350, 110])
#di default gurobi minimizza
model.setObjective(obj_coefs @ x, GRB.MAXIMIZE)

Risolviamo il problema

In [7]:
model.optimize()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.4.0 25E253)

CPU model: Apple M1 Pro
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 4 rows, 5 columns and 14 nonzeros (Max)
Model fingerprint: 0x98c1760a
Model has 5 linear objective coefficients
Coefficient statistics:
  Matrix range     [2e-01, 2e+00]
  Objective range  [1e+02, 4e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+01, 1e+02]

Presolve removed 1 rows and 1 columns
Presolve time: 0.01s
Presolved: 3 rows, 4 columns, 10 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.8333333e+04   2.500000e+00   0.000000e+00      0s
       1    1.7883333e+04   0.000000e+00   0.000000e+00      0s

Solved in 1 iterations and 0.01 seconds (0.00 work units)
Optimal objective  1.788333333e+04


Stampiamo per ogni variabile il valore nella soluzione ottima



In [8]:
for v in model.getVars():
  print('%s %g' % (v.VarName, v.X))

print('Obj: %g' % model.ObjVal)

C0 0
C1 10
C2 70.8333
C3 0
C4 70.8333
Obj: 17883.3


# Proviamo a risolvere il problema duale

In [9]:
dual_model = gp.Model("Duale")

lb_dual = [0.0, 0.0, -GRB.INFINITY, -GRB.INFINITY]
ub_dual = [GRB.INFINITY, GRB.INFINITY, 0.0, GRB.INFINITY]

y = dual_model.addMVar(A.shape[0], lb=lb_dual, ub=ub_dual, name="y")
dual_model.update()

dual_model.addConstr(A.T @ y >= obj_coefs)
dual_model.update()

dual_model.setObjective(b @ y, GRB.MINIMIZE)
dual_model.update()

model.optimize()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.4.0 25E253)

CPU model: Apple M1 Pro
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 4 rows, 5 columns and 14 nonzeros (Max)
Model has 5 linear objective coefficients
Coefficient statistics:
  Matrix range     [2e-01, 2e+00]
  Objective range  [1e+02, 4e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+01, 1e+02]


Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal objective  1.788333333e+04


# E' possibile accedere ai vincoli ed alla funzione obiettivo

In [10]:
for c in dual_model.getConstrs():
  print(dual_model.getRow(c))

2.0 y[0] + 0.5 y[1] + y[2]
1.5 y[0] + 0.25 y[1] + y[2]
0.5 y[0] + 0.25 y[1] + y[3]
2.5 y[0] + y[1]
0.7 y[0] + 0.3 y[1] + -1.0 y[3]


In [11]:
dual_model.getObjective()

<gurobi.LinExpr: 100.0 y[0] + 50.0 y[1] + 10.0 y[2]>

# Rendiamo il problema primale privo di soluzioni ammissibili

In [ ]:
#TODO

# Rendiamo il problema primale illimitato

In [3]:
model = gp.Model()
model.setParam(GRB.Param.DualReductions, 0)
n_vars = 5
#con lb definisco il lower bound e con ub l'upper bound, con [0] * n_vars ottengo una lista di zeri lunga n_vars 
x = model.addMVar(n_vars, lb=[0] * n_vars, ub=GRB.INFINITY)
model.update()
A =  np.array([[2., 1.5, 0.5, 2.5, 0.7],

               [0.5, 0.25, 0.25, 1, 0.3],
               [1, 1, 0, 0, 0],
               [0, 0, 1, 0, -1]])

b = np.array([100, 50, 10, 0])
ct2 = model.addConstr(A[-2]@x >= b[-2])
ct3 = model.addConstr(A[-1]@x == b[-1])

model.update()
obj_coefs = np.array([250, 230, 110, 350, 110])
#di default gurobi minimizza
model.setObjective(obj_coefs @ x, GRB.MAXIMIZE)
model.optimize()

Restricted license - for non-production use only - expires 2027-11-29
Set parameter DualReductions to value 0
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.4.0 25E253)

CPU model: Apple M1 Pro
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
DualReductions  0

Optimize a model with 2 rows, 5 columns and 4 nonzeros (Max)
Model fingerprint: 0x315ce0a7
Model has 5 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+02, 4e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+01, 1e+01]

Presolve time: 0.00s
Presolved: 1 rows, 4 columns, 2 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.0500000e+33   0.000000e+00   1.050000e+03      0s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Unbounded model


# Problema del cammino minimo
Dato un grafo $\mathcal{G}$ e due nodi $s,\ t$ vogliamo trovare il cammino che minimiza il costo di andare da $s$ a $t$.

Creiamo un grafo costituito da $200$ nodi

In [4]:
n_nodes = 200
adj_mat = np.random.rand(n_nodes, n_nodes)
adj_mat[adj_mat > 0.02] = 0
adj_mat *= 1000
adj_mat = np.round(adj_mat)
np.fill_diagonal(adj_mat, 0)

Definiamo il nodo di partenza ed il nodo di arrivo

In [5]:
s = 0
t = 197

## Risolviamo il problema con Gurobi

Se il costo di ogni arco e' positivo, il problema si puo' formulare nel seguente modo:
\begin{align}
  \text{minimize} &\sum_{ij} c_{ij}x_{ij}\\
                  & s.t.\\
                  0 \leq\ &x \leq 1\\
                  \sum_i x_{iv} + b_v &= \sum_j x_{vj},\ \forall v \in \text{Nodes}
\end{align}
dove $b_v$ il flusso in ingresso su ogni nodo ed $x_{ij}$ la quantita' di flusso nell'arco $(i, j)$.

In [6]:
import gurobipy as gp
from gurobipy import GRB

Creiamo il modello

In [7]:
m = gp.Model('shortestPath')

Aggiungiamo le variabili e definiamo la funzione obiettivo

In [8]:
var_idxs = np.nonzero(adj_mat)
var_idxs = [(i, j) for i, j in zip(var_idxs[0], var_idxs[1])]
costs = {idx: adj_mat[idx] for idx in var_idxs}

arcs = m.addVars(var_idxs, lb = 0, ub = 1, obj=costs, name="arcs")
m.update()

In [9]:
m.getObjective()

<gurobi.LinExpr: 3.0 arcs[0,20] + 16.0 arcs[0,79] + 13.0 arcs[1,20] + 11.0 arcs[1,118] + 4.0 arcs[1,145] + 17.0 arcs[2,120] + 15.0 arcs[2,137] + 3.0 arcs[2,144] + 13.0 arcs[2,155] + 8.0 arcs[3,13] + 17.0 arcs[3,27] + 10.0 arcs[3,47] + 15.0 arcs[3,74] + 2.0 arcs[3,75] + 9.0 arcs[3,135] + 15.0 arcs[3,155] + 19.0 arcs[4,0] + 8.0 arcs[4,3] + 11.0 arcs[4,22] + 2.0 arcs[4,50] + 16.0 arcs[4,56] + 18.0 arcs[4,98] + 9.0 arcs[4,135] + 17.0 arcs[4,196] + arcs[5,14] + 17.0 arcs[5,83] + 4.0 arcs[5,89] + arcs[5,101] + 13.0 arcs[5,102] + 4.0 arcs[5,124] + 17.0 arcs[5,139] + 16.0 arcs[5,162] + 9.0 arcs[6,33] + 17.0 arcs[6,36] + 6.0 arcs[6,37] + 7.0 arcs[7,2] + 15.0 arcs[7,16] + 18.0 arcs[7,43] + 10.0 arcs[7,89] + 15.0 arcs[7,91] + arcs[7,118] + 2.0 arcs[7,184] + 19.0 arcs[8,7] + 8.0 arcs[8,44] + 5.0 arcs[8,46] + 16.0 arcs[8,48] + 12.0 arcs[8,169] + 15.0 arcs[9,93] + 19.0 arcs[9,132] + 3.0 arcs[9,161] + 10.0 arcs[9,167] + 7.0 arcs[9,181] + 4.0 arcs[10,65] + 16.0 arcs[10,68] + 2.0 arcs[10,187] + 17.0 ar

Aggiungiamo i vincoli

In [10]:
b = np.zeros(n_nodes)
b[s] = 1
b[t] = -1

ct = m.addConstrs((arcs.sum('*', v) + b[v] == arcs.sum(v, '*') for v in range(n_nodes)))

Risolviamo il problema e stampiamo la soluzione trovata

In [11]:
m.optimize()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.4.0 25E253)

CPU model: Apple M1 Pro
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 200 rows, 777 columns and 1554 nonzeros (Min)
Model fingerprint: 0xce74bf3a
Model has 777 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 2e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Presolve removed 10 rows and 21 columns
Presolve time: 0.01s
Presolved: 190 rows, 756 columns, 1512 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0   -1.8000000e+01   7.000000e+00   0.000000e+00      0s
      26    3.7000000e+01   0.000000e+00   0.000000e+00      0s

Solved in 26 iterations and 0.01 seconds (0.00 work units)
Optimal objective  3.700000000e+01


In [12]:
print(f"Shortest path cost: {m.objVal:.4f}")
for v in var_idxs:
  if arcs[v].getAttr('X') == 1:
    print(f"{v[0]:3d} -> {v[1]:3d}: {arcs[v].getAttr('Obj'):.4f}")

Shortest path cost: 37.0000
  0 ->  20: 3.0000
 20 -> 151: 15.0000
 67 -> 142: 14.0000
142 -> 197: 1.0000
151 ->  67: 4.0000


# Misc Utili

In [ ]:
a = [1,2,3,4]
b = ["a", "b", "c", "d"]

## Itertools
Implementa molte funzioni utili per iterare su collezioni di elementi

In [ ]:
import itertools

In [ ]:
print(list(itertools.chain(a, b)))

[1, 2, 3, 4, 'a', 'b', 'c', 'd']


In [ ]:
print(list(itertools.product(a,b)))

[(1, 'a'), (1, 'b'), (1, 'c'), (1, 'd'), (2, 'a'), (2, 'b'), (2, 'c'), (2, 'd'), (3, 'a'), (3, 'b'), (3, 'c'), (3, 'd'), (4, 'a'), (4, 'b'), (4, 'c'), (4, 'd')]


## zip, enumerate

In [ ]:
for a_i, b_i in zip(a, b):
  print(a_i, b_i)

1 a
2 b
3 c
4 d


In [ ]:
for x in zip(a, b):
  print(x)

(1, 'a')
(2, 'b')
(3, 'c')
(4, 'd')


In [ ]:
for i, a_i in enumerate(a):
  print(i, a_i)

0 1
1 2
2 3
3 4


In [ ]:
for x in enumerate(a):
  print(x)

(0, 1)
(1, 2)
(2, 3)
(3, 4)


# Esercizio

Un polo industriale produce televisori utilizzando tre linee di montaggio (A, B e C) per televisori piatti, curvi e monitor ad alte prestazioni. Per stabilire la produzione per il prossimo semestre e’ necessario considerare I seguenti aspetti:

  * Dalla vendita dei tre prodotti l’azienda ricava un profitto pari a 200, 400 e 500 euro rispettivamente.

  * Nel corso dei sei mesi, la linea A potra’ essere impengata per un totale di 150 ore al mese, la linea B per 130 al mese e la linea C per 170 ore al mese. Tuttavia, esclusivamente per il quarto mese di produzione sarà possibile usufruire di ulteriori 20 ore di lavoro sulla linea A e di 10 ore in meno sulla linea C.

  * Produrre un televisore a schermo piatto richiede 18 ore sulla linea A e 7 sulla B, un televisore curvo invece ne richiede 12 sulla linea B e 25 sulla C, mentre un monitor ad alte prestazioni richiede 10 ore di lavorazione sulla linea A, 9 sulla linea B e 14 sulla linea C.

  * In un impianto separato l’azienda esegue verifiche sui prodotti ed ha una capacita’ lavorativa mensile pari a 120 ore. Per eseguire tutte le verifiche necessarie sono richieste 5 ore per un televisore a schermo piatto, 8 per uno curvo e 9 per un monitor.

  * A seguito di analisi di mercato vi sono I seguenti vincoli sulla produzione: nel secondo mese la produzione di monitor deve supereare le 10 unita’, nel terzo e quinto mese la produzione di televisori curvi deve essere almeno pari al 30% di quella di televisori piatti ed infine nel sesto mese di produzione non e’ prevista richiesta di monitor ad alte prestazioni.

  * Temendo uno sciopero dei lavoratori, la direzione considera che nel primo mese di produzione la capacita’ lavorativa dell’impianto di verifica dei prodotti sara’ ridotta al 75%.

  * la produzione complessiva sui 6 mesi di monitor ad alte prestazioni non deve superare le 40 unità

Sulla base di tali considerazioni, il problema e’ pianificiare la produzione del prossimo semestre con l’obiettivo di massimizzare il profitto dell’azienda.

In [2]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np

In [3]:
#per ogni mese (quindi 6 volte)
#z = max(pa*na + pb*nb + pc*nc)
#pa = 200
#pb = 400
#pc = 500
#vincoli:
#18*na + 10*nc <= 150 (+20)
#7*nb + 9*nb <= 130
#25*nb + 14*nc <= 170 (-10)
#5*na + 8*nb + 9*nc <= 120
#sum(nc) <= 40

#primo mese:
#5*na + 8*nb + 9*nc <= 120*0,25

#secondo mese:
#nc >= 11

#terzo mese:
#nb >= 0,30*na

#quarto mese:
#

#quinto mese:
#nb >= 0,30*na

#sesto mese:
#nc = 0

mesi = range(6)

profitto_piatti = 200
profitto_curvi = 400
profitto_monitor = 500

cap_A = [150] * 6
cap_B = [130] * 6
cap_C = [170] * 6
cap_verifica = [120] * 6

cap_A[3] += 20
cap_C[3] -= 10

cap_verifica[0] = 120 * 0.75

tempi_piatti = [18, 7, 0, 5]
tempi_curvi = [0, 12, 25, 8]
tempi_monitor = [10, 9, 14, 9]

model = gp.Model("tv")

#x_P[t] = numero di tv piatti prodotti nel mese t
#x_C[t] = numero di tv curvi prodotti nel mese t
#x_M[t] = numero di monitor prodotti nel mese t
x_P = model.addVars(mesi, vtype=GRB.INTEGER, name="Piatti")
x_C = model.addVars(mesi, vtype=GRB.INTEGER, name="Curvi")
x_M = model.addVars(mesi, vtype=GRB.INTEGER, name="Monitor")

model.setObjective(
    gp.quicksum(profitto_piatti * x_P[t] + 
                profitto_curvi * x_C[t] + 
                profitto_monitor * x_M[t] for t in mesi), 
    GRB.MAXIMIZE
)

for t in mesi:
    model.addConstr(tempi_piatti[0] * x_P[t] + tempi_curvi[0] * x_C[t] + tempi_monitor[0] * x_M[t] <= cap_A[t])
    
    model.addConstr(tempi_piatti[1] * x_P[t] + tempi_curvi[1] * x_C[t] + tempi_monitor[1] * x_M[t] <= cap_B[t])
    
    model.addConstr(tempi_piatti[2] * x_P[t] + tempi_curvi[2] * x_C[t] + tempi_monitor[2] * x_M[t] <= cap_C[t])
    
    model.addConstr(tempi_piatti[3] * x_P[t] + tempi_curvi[3] * x_C[t] + tempi_monitor[3] * x_M[t] <= cap_verifica[t])

#secondo mese: monitor > 10
model.addConstr(x_M[1] >= 11)

#terzo mese e quinto mese: curvi >= 30% dei piatti
model.addConstr(x_C[2] >= 0.3 * x_P[2])
model.addConstr(x_C[4] >= 0.3 * x_P[4])

#sesto mese: 0 monitor
model.addConstr(x_M[5] == 0)

model.optimize()

if model.status == GRB.OPTIMAL:
    for t in mesi:
        print(f"Mese {t+1}:")
        print(f"  TV Piatti:  {int(x_P[t].X)}")
        print(f"  TV Curvi:   {int(x_C[t].X)}")
        print(f"  Monitor:    {int(x_M[t].X)}")
else:
    print("Il modello non ha trovato una soluzione ottimale")

Restricted license - for non-production use only - expires 2027-11-29
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.4.0 25E253)



CPU model: Apple M1 Pro
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 28 rows, 18 columns and 66 nonzeros (Max)
Model fingerprint: 0x3dfcecf7
Model has 18 linear objective coefficients
Variable types: 0 continuous, 18 integer (0 binary)
Coefficient statistics:
  Matrix range     [3e-01, 2e+01]
  Objective range  [2e+02, 5e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+01, 2e+02]

Found heuristic solution: objective 23500.000000
Presolve removed 23 rows and 15 columns
Presolve time: 0.01s
Presolved: 5 rows, 3 columns, 12 nonzeros
Found heuristic solution: objective 27100.000000
Variable types: 0 continuous, 3 integer (0 binary)
Found heuristic solution: objective 29900.000000

Root relaxation: objective 3.361225e+04, 2 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0  